# LeetCode 2130 — Maximum Twin Sum of a Linked List bằng Python

Link đề: https://leetcode.com/problems/maximum-twin-sum-of-a-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List, Two Pointers, Stack

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def pairSum(self, head: Optional[ListNode]) -> int:
        ...
```

## 1. Mục tiêu học

1. Tự dựng `ListNode`, hàm **dựng danh sách từ list Python** và **xuất ngược lại** để test như LeetCode.
2. Hiểu vì sao danh sách liên kết đơn **không truy cập ngẫu nhiên** được và cái giá của việc "đi tới nút thứ k".
3. Cài đặt bốn cách: sao chép ra mảng + hai con trỏ, ngăn xếp, **con trỏ nhanh/chậm + đảo nửa sau** (O(1) bộ nhớ), và **đảo nửa đầu trong lúc tìm giữa** (một lượt).
4. Thành thạo kỹ thuật **đảo danh sách liên kết** bằng vòng lặp (không đệ quy) và biết **khôi phục** lại danh sách nếu cần.
5. Viết test: kiểm tra kết quả, kiểm tra danh sách sau khi chạy (có bị phá không).

## 2. Đề bài & ví dụ

Danh sách liên kết có `n` nút, **`n` chẵn**. Nút thứ `i` (đánh số từ 0) là **sinh đôi** (twin) của nút `n − 1 − i`, với `0 ≤ i ≤ n/2 − 1`. Ví dụ `n = 4`: nút 0 ↔ nút 3, nút 1 ↔ nút 2.

**Tổng sinh đôi** = giá trị nút + giá trị nút sinh đôi của nó. Trả về **tổng sinh đôi lớn nhất**.

**Ví dụ 1:** `head = [5,4,2,1]` → `6` (5+1 = 6, 4+2 = 6).

**Ví dụ 2:** `head = [4,2,2,3]` → `7` (4+3 = 7, 2+2 = 4).

**Ví dụ 3:** `head = [1,100000]` → `100001`.

**Ràng buộc:** số nút là số chẵn trong `[2, 10⁵]`; `1 ≤ Node.val ≤ 10⁵`.

In [1]:
from typing import List, Optional
import random, time

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    def __repr__(self):
        return f"ListNode({self.val})"

def build(vals):
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head, limit=10**6):
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

examples = [([5, 4, 2, 1], 6), ([4, 2, 2, 3], 7), ([1, 100000], 100001)]
for vals, exp in examples:
    h = build(vals)
    print(" -> ".join(map(str, to_list(h))), "| cặp sinh đôi:",
          [(vals[i], vals[-1 - i]) for i in range(len(vals) // 2)], "| kỳ vọng", exp)

5 -> 4 -> 2 -> 1 | cặp sinh đôi: [(5, 1), (4, 2)] | kỳ vọng 6
4 -> 2 -> 2 -> 3 | cặp sinh đôi: [(4, 3), (2, 2)] | kỳ vọng 7
1 -> 100000 | cặp sinh đôi: [(1, 100000)] | kỳ vọng 100001


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn

Mỗi nút giữ `val` và con trỏ `next` tới nút kế. Chỉ có `head` trong tay; muốn tới nút thứ `k` phải đi `k` bước — **O(k)**, không có `head[k]`. Nút cuối có `next = None`.

In [2]:
def kth(head, k):
    for _ in range(k):
        head = head.next
    return head

h = build([5, 4, 2, 1])
print(kth(h, 0), kth(h, 3), kth(h, 3).next)

ListNode(5) ListNode(1) None


### 3.2 Nút giả (dummy) khi dựng danh sách

Dùng `dummy = ListNode()` làm "đầu giả" để khỏi xử lý riêng trường hợp danh sách rỗng; cuối cùng trả `dummy.next`. Hàm `build` ở trên dùng đúng kỹ thuật này.

### 3.3 Đảo danh sách liên kết bằng vòng lặp

Ba biến `prev, cur, nxt`: mỗi bước "bẻ" con trỏ `cur.next` quay về `prev`. Không dùng đệ quy vì `n = 10⁵` sẽ vượt giới hạn đệ quy mặc định của Python (1000).

In [3]:
def reverse(head):
    prev = None
    cur = head
    while cur:
        nxt = cur.next      # 1. nhớ nút kế
        cur.next = prev     # 2. bẻ con trỏ quay ngược
        prev = cur          # 3. tiến prev
        cur = nxt           # 4. tiến cur
    return prev             # đầu mới

print(to_list(reverse(build([1, 2, 3, 4]))))
import sys; print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

[4, 3, 2, 1]
giới hạn đệ quy mặc định: 1000


### 3.4 Con trỏ nhanh/chậm tìm điểm giữa

`slow` đi 1 bước, `fast` đi 2 bước. Với `n` chẵn, nếu bắt đầu `slow = fast = head` và lặp `while fast and fast.next`, khi dừng `slow` là nút **thứ `n/2`** (đầu nửa sau).

In [4]:
def middle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

for vals in [[1, 2], [1, 2, 3, 4], [1, 2, 3, 4, 5, 6]]:
    print(vals, "-> đầu nửa sau:", middle(build(vals)).val)

[1, 2] -> đầu nửa sau: 2
[1, 2, 3, 4] -> đầu nửa sau: 3
[1, 2, 3, 4, 5, 6] -> đầu nửa sau: 4


## 4. Ý tưởng nền tảng / trực giác

Viết các giá trị thành một hàng và **gập đôi** ở giữa:

```
[4, 2, 2, 3]       gập      4  2
                    ->       3  2      tổng từng cột: 7, 4  -> max = 7
```

Nói cách khác, ta cần ghép nút thứ `i` từ **đầu** với nút thứ `i` từ **cuối**. Khó khăn duy nhất: danh sách đơn chỉ đi được **xuôi**, không đi ngược từ cuối.

Ba cách "đi ngược" có thể dùng:
1. Chép ra mảng (mảng thì truy cập hai đầu thoải mái).
2. Ngăn xếp: đẩy nửa đầu vào, khi đi nửa sau thì pop ra — pop trả về theo thứ tự ngược.
3. **Đảo** nửa sau (hoặc nửa đầu) ngay trên danh sách, rồi đi song song hai nửa.

Làm tay ví dụ 1 với cách 3: `5 → 4 → 2 → 1`. Nửa sau `2 → 1` đảo thành `1 → 2`. Đi song song: `(5, 1)`, `(4, 2)` ⇒ tổng 6, 6 ⇒ đáp án 6.

In [5]:
vals = [4, 2, 2, 3]
half = len(vals) // 2
first, second = vals[:half], vals[half:][::-1]
print("nửa đầu       :", first)
print("nửa sau (đảo) :", second)
print("tổng từng cặp :", [a + b for a, b in zip(first, second)], "-> max =", max(a + b for a, b in zip(first, second)))

nửa đầu       : [4, 2]
nửa sau (đảo) : [3, 2]
tổng từng cặp : [7, 4] -> max = 7


## 5. Cách 1 — Brute Force / baseline: với mỗi `i`, đi tới nút `n − 1 − i`

**Ý tưởng:** đếm `n`; với mỗi nút `i` của nửa đầu, đi từ `head` tới nút `n − 1 − i` để lấy giá trị sinh đôi. Không cần bộ nhớ phụ nhưng chậm.

In [6]:
def brute(head):
    n = 0
    cur = head
    while cur:
        n += 1; cur = cur.next
    best = 0
    a = head
    for i in range(n // 2):
        b = head
        for _ in range(n - 1 - i):        # đi tới nút sinh đôi
            b = b.next
        best = max(best, a.val + b.val)
        a = a.next
    return best

for vals, exp in examples:
    got = brute(build(vals))
    print(got, "OK" if got == exp else "SAI")

6 OK
7 OK
100001 OK


**Độ phức tạp:** mỗi `i` đi `n − 1 − i` bước ⇒ tổng ≈ `3n²/8` ⇒ **O(n²)** thời gian, O(1) bộ nhớ. Với `n = 10⁵`: ~4·10⁹ bước ⇒ quá chậm.

**Nút thắt:** lặp đi lặp lại việc đi từ đầu danh sách. Mỗi lần ta chỉ cần nút sinh đôi **kế tiếp** (lùi một nút so với lần trước) — điều mà danh sách đơn không làm được nếu không có thêm cấu trúc.

In [7]:
for n in [500, 2000]:
    h = build(list(range(1, n + 1)))
    t = time.perf_counter(); brute(h); dt = time.perf_counter() - t
    print(f"n={n}: {dt*1000:.0f} ms")

n=500: 2 ms
n=2000: 24 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** ta cần duyệt nửa sau theo **thứ tự ngược**. Có hai cách để có "thứ tự ngược" mà không tốn O(n) mỗi bước:
- **Lưu** lại (mảng/ngăn xếp) ⇒ O(n) bộ nhớ.
- **Đảo** con trỏ của một nửa ⇒ O(1) bộ nhớ, nhưng **sửa** danh sách (có thể khôi phục sau).

**Bất biến cho cách đảo nửa sau:** sau khi tìm `mid` (đầu nửa sau) và đảo nửa sau thành `q`, ta có hai chuỗi `p = head` (nửa đầu, n/2 nút) và `q` (nửa sau đảo, n/2 nút). Ở bước thứ `k`, `p` là nút `k` và `q` là nút `n − 1 − k` ⇒ `p.val + q.val` đúng là tổng sinh đôi thứ `k`.

**Quy tắc dừng:** vì nửa đầu vẫn trỏ vào `mid` (nút đầu nửa sau, giờ là **đuôi** của `q`), ta dừng theo `q` (`while q`), không theo `p`.

**Biến thể một lượt:** trong lúc `fast` chạy để tìm giữa, `slow` vừa đi vừa **đảo nửa đầu**. Khi `fast` tới cuối, `prev` là đầu nửa đầu đã đảo (nút `n/2 − 1, …, 0`) và `slow` là đầu nửa sau (nút `n/2, …, n − 1`) ⇒ đi song song.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chép ra mảng + hai con trỏ

In [8]:
def to_array(head):
    vals = []
    while head:
        vals.append(head.val)
        head = head.next
    n = len(vals)
    return max(vals[i] + vals[n - 1 - i] for i in range(n // 2))

for vals, exp in examples:
    got = to_array(build(vals))
    print(got, "OK" if got == exp else "SAI")

6 OK
7 OK
100001 OK


**Độ phức tạp:** O(n) thời gian, O(n) bộ nhớ. Đơn giản, khó sai — rất nên dùng trong phỏng vấn nếu không bị yêu cầu O(1) bộ nhớ.

Code nộp LeetCode (cách 2):

In [9]:
class SolutionArray:
    def pairSum(self, head: Optional[ListNode]) -> int:
        vals = []
        while head:
            vals.append(head.val)
            head = head.next
        i, j = 0, len(vals) - 1
        best = 0
        while i < j:
            best = max(best, vals[i] + vals[j])
            i += 1; j -= 1
        return best

print([SolutionArray().pairSum(build(v)) for v, _ in examples])

[6, 7, 100001]


### 7.2 Cách 3 — Ngăn xếp với con trỏ nhanh/chậm

`slow` đi nửa đầu và **đẩy** giá trị vào ngăn xếp (`fast` chạy gấp đôi để biết lúc nào hết nửa đầu). Sau đó `slow` đi nửa sau, mỗi bước **pop** ra giá trị sinh đôi. Không cần đếm `n` trước.

In [10]:
def stack_half(head):
    stack = []
    slow = fast = head
    while fast and fast.next:
        stack.append(slow.val)
        slow = slow.next
        fast = fast.next.next
    best = 0
    while slow:
        best = max(best, slow.val + stack.pop())
        slow = slow.next
    return best

for vals, exp in examples:
    got = stack_half(build(vals))
    print(got, "OK" if got == exp else "SAI")

6 OK
7 OK
100001 OK


**Độ phức tạp:** O(n) thời gian, O(n/2) bộ nhớ. Chỉ lưu nửa đầu.

### 7.3 Cách 4 — Đảo nửa sau tại chỗ (O(1) bộ nhớ) + khôi phục

Ba bước: (1) tìm `mid`, (2) đảo nửa sau, (3) đi song song tính max. Bước (4) tùy chọn: đảo lại nửa sau để **khôi phục** danh sách cho người gọi.

In [11]:
def reverse_second_half(head, restore=True):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    # slow = đầu nửa sau
    q = reverse(slow)
    best = 0
    p, t = head, q
    while t:
        best = max(best, p.val + t.val)
        p, t = p.next, t.next
    if restore:
        reverse(q)          # đảo lại; nửa đầu vẫn trỏ vào slow nên danh sách liền lại
    return best

for vals, exp in examples:
    h = build(vals)
    got = reverse_second_half(h)
    print(got, "OK" if got == exp else "SAI", "| danh sách sau khi chạy:", to_list(h))

6 OK | danh sách sau khi chạy: [5, 4, 2, 1]
7 OK | danh sách sau khi chạy: [4, 2, 2, 3]
100001 OK | danh sách sau khi chạy: [1, 100000]


Vì sao khôi phục chỉ cần `reverse(q)`? Khi đảo nửa sau, nút cuối nửa đầu vẫn trỏ vào `slow` (nút đầu nửa sau cũ), còn `slow.next` thành `None`. Đảo `q` lần nữa đưa nửa sau về thứ tự cũ với đầu là `slow` — và nửa đầu vẫn đang trỏ vào `slow` ⇒ danh sách nối liền như ban đầu.

Code nộp LeetCode (cách 4, không cần khôi phục):

In [12]:
class SolutionReverse:
    def pairSum(self, head: Optional[ListNode]) -> int:
        slow = fast = head
        while fast and fast.next:
            slow, fast = slow.next, fast.next.next
        prev = None
        while slow:                      # đảo nửa sau
            slow.next, prev, slow = prev, slow, slow.next
        best = 0
        while prev:
            best = max(best, head.val + prev.val)
            head, prev = head.next, prev.next
        return best

print([SolutionReverse().pairSum(build(v)) for v, _ in examples])

[6, 7, 100001]


**Cẩn thận với phép gán đồng thời** `slow.next, prev, slow = prev, slow, slow.next`: vế phải được tính **hết** trước (lấy `prev`, `slow`, `slow.next` cũ), rồi mới gán từ trái sang phải. Thứ tự ở vế trái quan trọng: `slow.next` phải được gán **trước** khi `slow` bị đổi. Mục 13 có ví dụ viết sai thứ tự.

**Độ phức tạp cách 4:** O(n) thời gian, **O(1)** bộ nhớ phụ.

### 7.4 Cách 5 — Đảo nửa đầu trong lúc tìm giữa (một lượt)

In [13]:
def reverse_first_half_one_pass(head):
    prev = None
    slow = fast = head
    while fast and fast.next:
        fast = fast.next.next            # fast đi trước (vì slow.next sắp bị bẻ)
        nxt = slow.next
        slow.next = prev                 # đảo nửa đầu
        prev, slow = slow, nxt
    # prev: nửa đầu đã đảo (nút n/2-1 ... 0); slow: nửa sau (nút n/2 ... n-1)
    best = 0
    while slow:
        best = max(best, prev.val + slow.val)
        prev, slow = prev.next, slow.next
    return best

for vals, exp in examples:
    got = reverse_first_half_one_pass(build(vals))
    print(got, "OK" if got == exp else "SAI")

6 OK
7 OK
100001 OK


Ở đây `fast` phải tiến **trước** khi bẻ `slow.next`, vì khi `slow` là `head` thì `fast` cũng là `head` — bẻ `slow.next` trước sẽ làm `fast.next` thành `None`.

Cặp ghép đúng? Sau vòng lặp, `prev` là nút `n/2 − 1` (đi lùi về 0), `slow` là nút `n/2` (đi tới `n − 1`). Bước `k`: `prev` = nút `n/2 − 1 − k`, `slow` = nút `n/2 + k`; tổng chỉ số = `n − 1` ✔ — đúng là cặp sinh đôi.

**Độ phức tạp cách 5:** O(n) thời gian (một lượt tìm giữa + một lượt nửa), O(1) bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 4 trên `[4, 2, 2, 3]`

| bước | slow | fast | ghi chú |
|---|---|---|---|
| 0 | 4 (nút 0) | 4 (nút 0) | bắt đầu |
| 1 | 2 (nút 1) | 2 (nút 2) | |
| 2 | 2 (nút 2) | None | dừng: slow = đầu nửa sau |

Đảo nửa sau `2(nút 2) → 3` thành `3 → 2(nút 2)`. Đi song song: `(4, 3) = 7`, `(2, 2) = 4` ⇒ 7.

In [14]:
def trace_reverse_second(vals):
    head = build(vals)
    slow = fast = head; step = 0
    print(f"bước {step}: slow={slow.val}, fast={fast.val}")
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next; step += 1
        print(f"bước {step}: slow={slow.val}, fast={fast.val if fast else None}")
    print("nửa sau trước khi đảo:", to_list(slow))
    q = reverse(slow)
    print("nửa sau sau khi đảo  :", to_list(q))
    print("nửa đầu (vẫn trỏ vào nút giữa cũ):", to_list(head))
    p, t, best = head, q, 0
    while t:
        best = max(best, p.val + t.val)
        print(f"  cặp ({p.val}, {t.val}) -> tổng {p.val + t.val}, best = {best}")
        p, t = p.next, t.next
    return best

trace_reverse_second([4, 2, 2, 3])

bước 0: slow=4, fast=4
bước 1: slow=2, fast=2
bước 2: slow=2, fast=None
nửa sau trước khi đảo: [2, 3]
nửa sau sau khi đảo  : [3, 2]
nửa đầu (vẫn trỏ vào nút giữa cũ): [4, 2, 2]
  cặp (4, 3) -> tổng 7, best = 7
  cặp (2, 2) -> tổng 4, best = 7


7

### 8.2 Trace cách 5 (đảo nửa đầu một lượt) trên `[1, 2, 3, 4, 5, 6]`

In [15]:
def trace_one_pass(vals):
    head = build(vals)
    prev = None; slow = fast = head
    while fast and fast.next:
        fast = fast.next.next
        nxt = slow.next; slow.next = prev; prev, slow = slow, nxt
        print(f"đảo xong tới nút {prev.val}: nửa đầu đảo = {to_list(prev)}, slow = {slow.val}, fast = {fast.val if fast else None}")
    best = 0
    while slow:
        best = max(best, prev.val + slow.val)
        print(f"  cặp ({prev.val}, {slow.val}) = {prev.val + slow.val}")
        prev, slow = prev.next, slow.next
    print("best =", best)

trace_one_pass([1, 2, 3, 4, 5, 6])

đảo xong tới nút 1: nửa đầu đảo = [1], slow = 2, fast = 3
đảo xong tới nút 2: nửa đầu đảo = [2, 1], slow = 3, fast = 5
đảo xong tới nút 3: nửa đầu đảo = [3, 2, 1], slow = 4, fast = None
  cặp (3, 4) = 7
  cặp (2, 5) = 7
  cặp (1, 6) = 7
best = 7


### 8.3 Trace cách 3 (ngăn xếp)

In [16]:
def trace_stack(vals):
    head = build(vals); stack = []; slow = fast = head
    while fast and fast.next:
        stack.append(slow.val); slow = slow.next; fast = fast.next.next
        print("đẩy -> stack =", stack)
    best = 0
    while slow:
        top = stack.pop(); best = max(best, slow.val + top)
        print(f"nút {slow.val} + pop {top} = {slow.val + top}, best = {best}, stack = {stack}")
        slow = slow.next

trace_stack([5, 4, 2, 1])

đẩy -> stack = [5]
đẩy -> stack = [5, 4]
nút 2 + pop 4 = 6, best = 6, stack = [5]
nút 1 + pop 5 = 6, best = 6, stack = []


## 9. Code nộp LeetCode

Bản O(1) bộ nhớ (đảo nửa sau). Trên LeetCode lớp `ListNode` và `Optional` đã có sẵn.

In [17]:
from typing import Optional

class Solution:
    def pairSum(self, head: Optional[ListNode]) -> int:
        # 1) tìm đầu nửa sau
        slow = fast = head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
        # 2) đảo nửa sau
        prev = None
        while slow:
            nxt = slow.next
            slow.next = prev
            prev = slow
            slow = nxt
        # 3) ghép nửa đầu với nửa sau đã đảo
        best = 0
        p, q = head, prev
        while q:
            best = max(best, p.val + q.val)
            p = p.next
            q = q.next
        return best

In [18]:
sol = Solution()
for vals, exp in examples:
    got = sol.pairSum(build(vals))
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

6 | kỳ vọng 6 | OK
7 | kỳ vọng 7 | OK
100001 | kỳ vọng 100001 | OK


## 10. Vì sao thuật toán đúng

**Tìm giữa:** sau `k` vòng, `slow` là nút `k`, `fast` là nút `2k`. Vòng lặp tiếp tục khi nút `2k` và `2k + 1` tồn tại, tức `2k + 1 ≤ n − 1`. Với `n` chẵn, vòng dừng khi `2k = n` (fast = None) ⇒ `k = n/2` ⇒ `slow` là nút `n/2` — đầu nửa sau. ✔

**Đảo:** bất biến của vòng đảo: `prev` là đầu của phần đã đảo (các nút đã xét, theo thứ tự ngược), `slow` là đầu phần chưa xét. Kết thúc: `prev` là nút `n − 1`, đi tiếp là `n − 2, …, n/2`.

**Ghép:** bước `k` (k = 0..n/2 − 1): `p` là nút `k`, `q` là nút `n − 1 − k` ⇒ `p.val + q.val` là tổng sinh đôi thứ `k`. `q` có đúng `n/2` nút nên vòng lặp chạy đúng `n/2` lần và `p` không bao giờ là `None` trong lúc đó. Lấy max trên mọi `k` ⇒ đáp án. ∎

Khởi tạo `best = 0` an toàn vì mọi giá trị ≥ 1 nên mọi tổng ≥ 2 > 0.

## 11. Phân tích Time / Space Complexity

`n` = số nút (chẵn, ≤ 10⁵).

| Cách | Time | Space phụ | Sửa danh sách? |
|---|---|---|---|
| 1. Đi tới nút sinh đôi | O(n²) | O(1) | không |
| 2. Chép ra mảng | O(n) | O(n) | không |
| 3. Ngăn xếp nửa đầu | O(n) | O(n/2) | không |
| 4. Đảo nửa sau | O(n) | O(1) | có (khôi phục được) |
| 5. Đảo nửa đầu một lượt | O(n) | O(1) | có |

Đo trên `n = 10⁵`:

In [19]:
rng = random.Random(2130)
vals = [rng.randint(1, 10**5) for _ in range(10**5)]
ref = max(vals[i] + vals[-1 - i] for i in range(len(vals) // 2))
for name, f in [("to_array", to_array), ("stack_half", stack_half), ("reverse_second_half", reverse_second_half),
                ("one_pass", reverse_first_half_one_pass), ("Solution", Solution().pairSum)]:
    h = build(vals)
    t = time.perf_counter(); r = f(h); dt = time.perf_counter() - t
    assert r == ref
    print(f"{name:<20} {dt*1000:.1f} ms")

to_array             6.4 ms


stack_half           4.7 ms
reverse_second_half  6.7 ms


one_pass             8.0 ms


Solution             6.4 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "to_array": to_array, "SolutionArray": SolutionArray().pairSum,
       "stack_half": stack_half, "reverse_second_half": reverse_second_half,
       "SolutionReverse": SolutionReverse().pairSum, "one_pass": reverse_first_half_one_pass,
       "Solution": Solution().pairSum}
edge = examples + [
    ([1, 1], 2),
    ([1, 2, 3, 4], 5),
    ([10, 1, 1, 1, 1, 10], 20),
    ([1, 100, 1, 1, 1, 1], 101),
    ([3, 3, 3, 3, 3, 3, 3, 3], 6),
    ([100000] * 6, 200000),
]
for vals, exp in edge:
    for name, f in ALL.items():
        assert f(build(vals)) == exp, (name, vals, exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


### 12.2 Đối chiếu ngẫu nhiên + kiểm tra danh sách không bị phá (với các cách không sửa / có khôi phục)

In [21]:
rng = random.Random(42)
non_destructive = ["brute", "to_array", "SolutionArray", "stack_half", "reverse_second_half"]
for _ in range(2000):
    n = 2 * rng.randint(1, 20)
    vals = [rng.randint(1, rng.choice([5, 100, 10**5])) for _ in range(n)]
    ref = max(vals[i] + vals[n - 1 - i] for i in range(n // 2))
    for name, f in ALL.items():
        h = build(vals)
        assert f(h) == ref, (name, vals)
        if name in non_destructive:
            assert to_list(h) == vals, (name, "danh sách bị thay đổi")
print("2000 case OK; các cách không phá danh sách đã được kiểm tra")

2000 case OK; các cách không phá danh sách đã được kiểm tra


### 12.3 Các cách đảo **có** làm thay đổi danh sách — kiểm tra để biết

In [22]:
for name in ["SolutionReverse", "one_pass", "Solution"]:
    h = build([1, 2, 3, 4, 5, 6])
    ALL[name](h)
    print(f"{name:<16} -> danh sách từ head sau khi chạy: {to_list(h, limit=20)}")

SolutionReverse  -> danh sách từ head sau khi chạy: [1, 2, 3, 4]
one_pass         -> danh sách từ head sau khi chạy: [1]
Solution         -> danh sách từ head sau khi chạy: [1, 2, 3, 4]


Trên LeetCode điều này chấp nhận được, nhưng trong code thật nên khôi phục (như `reverse_second_half(restore=True)`) hoặc ghi rõ trong tài liệu hàm.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dừng vòng ghép theo `p` thay vì `q` (đọc lấn sang nửa sau)

In [23]:
def wrong_stop_on_p(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    q = reverse(slow); p = head; best = 0
    try:
        while p:                                 # SAI: p đi qua cả nút giữa cũ
            best = max(best, p.val + q.val)
            p, q = p.next, q.next
        return best
    except AttributeError as e:
        return f"AttributeError: {e}"

print("bản sai :", wrong_stop_on_p(build([4, 2, 2, 3])))
print("bản đúng:", Solution().pairSum(build([4, 2, 2, 3])))

bản sai : AttributeError: 'NoneType' object has no attribute 'val'
bản đúng: 7


**Vì sao sai:** nửa đầu vẫn nối vào nút giữa cũ (nay là đuôi của `q`), nên `p` dài `n/2 + 1` nút còn `q` chỉ `n/2` nút ⇒ `q` thành `None` trước ⇒ lỗi `NoneType`. **Sửa:** `while q:`.

### Lỗi 2 — Sai thứ tự trong phép gán đồng thời khi đảo

In [24]:
def wrong_tuple_assign(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    prev = None
    steps = 0
    try:
        while slow and steps < 10:
            prev, slow, slow.next = slow, slow.next, prev     # SAI: slow đổi trước khi gán slow.next
            steps += 1
    except AttributeError as e:
        return f"AttributeError ở bước {steps}: {e}; nửa sau còn lại từ prev = {to_list(prev, limit=10)}"
    return to_list(prev, limit=10)

print("bản sai (nửa sau đảo của [1,2,3,4,5,6]):", wrong_tuple_assign(build([1, 2, 3, 4, 5, 6])))
print("bản đúng                              :", to_list(reverse(build([4, 5, 6]))))

bản sai (nửa sau đảo của [1,2,3,4,5,6]): AttributeError ở bước 1: 'NoneType' object has no attribute 'next' and no __dict__ for setting new attributes; nửa sau còn lại từ prev = [5]
bản đúng                              : [6, 5, 4]


**Vì sao sai:** vế phải tính xong `(slow, slow.next, prev)` cũ, nhưng vế trái gán `prev`, rồi `slow` (đã thành nút kế), rồi `slow.next = prev` — gán vào **nút kế** chứ không phải nút hiện tại ⇒ cấu trúc hỏng; tới nút cuối thì `slow` thành `None` và `None.next = …` gây `AttributeError` (ta bắt lỗi để in ra). **Sửa:** viết bốn dòng `nxt = …` tách bạch, hoặc đặt `slow.next` **đầu tiên** ở vế trái.

### Lỗi 3 — Cách 5: bẻ `slow.next` trước khi `fast` tiến

In [25]:
def wrong_one_pass_order(head):
    prev = None; slow = fast = head
    while fast and fast.next:
        nxt = slow.next
        slow.next = prev                 # SAI: khi slow is fast, fast.next vừa bị bẻ
        fast = fast.next.next if fast.next else None
        prev, slow = slow, nxt
    best = 0
    while slow and prev:
        best = max(best, prev.val + slow.val)
        prev, slow = prev.next, slow.next
    return best

print("bản sai :", wrong_one_pass_order(build([1, 2, 3, 4, 5, 6])))
print("bản đúng:", reverse_first_half_one_pass(build([1, 2, 3, 4, 5, 6])))

bản sai : 3
bản đúng: 7


**Vì sao sai:** ở vòng đầu `slow` và `fast` cùng là `head`; bẻ `head.next = None` làm `fast` không tiến được, vòng lặp dừng sớm và hai nửa bị lệch. **Sửa:** cập nhật `fast` **trước**.

### Lỗi 4 — Dùng đệ quy để đảo danh sách (RecursionError với n lớn)

In [26]:
def reverse_recursive(node):
    if node is None or node.next is None:
        return node
    new_head = reverse_recursive(node.next)
    node.next.next = node
    node.next = None
    return new_head

try:
    reverse_recursive(build(list(range(5000))))
    print("không lỗi")
except RecursionError as e:
    print("bản đệ quy -> RecursionError:", str(e)[:45])
print("bản lặp:", to_list(reverse(build(list(range(5000)))))[:3], "...")

bản đệ quy -> RecursionError: maximum recursion depth exceeded
bản lặp: [4999, 4998, 4997] ...


**Sửa:** luôn đảo bằng vòng lặp khi `n` có thể tới 10⁵.

### Lỗi 5 — Ghép nút `i` với nút `n − i` (lệch một)

In [27]:
def wrong_off_by_one(head):
    vals = to_list(head); n = len(vals)
    return max(vals[i] + vals[(n - i) % n] for i in range(n // 2))   # SAI: phải là n-1-i

print("bản sai :", wrong_off_by_one(build([4, 2, 2, 3])))
print("bản đúng:", Solution().pairSum(build([4, 2, 2, 3])))

bản sai : 8
bản đúng: 7


**Sửa:** sinh đôi của `i` là `n − 1 − i` (hoặc `vals[-1 - i]` / `vals[~i]` trong Python).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ghép phần tử thứ i từ đầu với thứ i từ cuối" trên linked list ⇒ **tìm giữa + đảo nửa** (giống LC 234 Palindrome Linked List, LC 143 Reorder List).
- Con trỏ nhanh/chậm: nhớ hai biến thể điều kiện `while fast and fast.next` (slow dừng ở giữa-phải) và `while fast.next and fast.next.next` (giữa-trái).
- Đảo danh sách: thuộc lòng bốn dòng `nxt = cur.next; cur.next = prev; prev = cur; cur = nxt`.
- Nếu đề không yêu cầu O(1) bộ nhớ ⇒ chép ra mảng là lời giải an toàn nhất.
- Mẹo Python: `vals[~i]` chính là `vals[-1 - i]` (toán tử `~i = -i - 1`).

In [28]:
vals = [4, 2, 2, 3]
print([(vals[i], vals[~i]) for i in range(len(vals) // 2)], "~0 =", ~0, "~1 =", ~1)

[(4, 3), (2, 2)] ~0 = -1 ~1 = -2


## 15. Biến thể của bài

### 15.1 Biến thể A — Kiểm tra danh sách liên kết **đối xứng** (LC 234)

Cùng khung: tìm giữa, đảo nửa sau, so sánh từng cặp. Với `n` lẻ, nút giữa không cần so.

In [29]:
def is_palindrome(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    q = reverse(slow); p = head; ok = True; t = q
    while t:
        if p.val != t.val: ok = False; break
        p, t = p.next, t.next
    reverse(q)                       # khôi phục
    return ok

for vals in [[1, 2, 2, 1], [1, 2, 3, 2, 1], [1, 2], [7]]:
    h = build(vals)
    print(vals, is_palindrome(h), "| sau khi chạy:", to_list(h))

[1, 2, 2, 1] True | sau khi chạy: [1, 2, 2, 1]
[1, 2, 3, 2, 1] True | sau khi chạy: [1, 2, 3, 2, 1]
[1, 2] False | sau khi chạy: [1, 2]
[7] True | sau khi chạy: [7]


### 15.2 Biến thể B — Trả về **mảng mọi tổng sinh đôi**, và **sắp xếp lại** danh sách kiểu L0 → Ln−1 → L1 → Ln−2 … (LC 143)

In [30]:
def twin_sums(head):
    vals = to_list(head)
    return [vals[i] + vals[~i] for i in range(len(vals) // 2)]

def reorder(head):
    if not head or not head.next: return head
    slow = fast = head
    while fast.next and fast.next.next:          # slow = giữa-trái
        slow, fast = slow.next, fast.next.next
    second = reverse(slow.next); slow.next = None
    p, q = head, second
    while q:
        pn, qn = p.next, q.next
        p.next = q; q.next = pn
        p, q = pn, qn
    return head

print(twin_sums(build([4, 2, 2, 3])))
for vals in [[1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]:
    print(vals, "->", to_list(reorder(build(vals))))

[7, 4]
[1, 2, 3, 4] -> [1, 4, 2, 3]
[1, 2, 3, 4, 5] -> [1, 5, 2, 4, 3]
[1, 2, 3, 4, 5, 6] -> [1, 6, 2, 5, 3, 4]


### 15.3 Biến thể C — Độ dài lẻ: nút giữa không có sinh đôi, hoặc tính như "tự sinh đôi" (`2·val`)

In [31]:
def pair_sum_odd(head, middle_self=True):
    vals = to_list(head); n = len(vals)
    cands = [vals[i] + vals[~i] for i in range(n // 2)]
    if n % 2 and middle_self:
        cands.append(2 * vals[n // 2])
    return max(cands) if cands else None

print(pair_sum_odd(build([1, 9, 2])), pair_sum_odd(build([1, 9, 2]), middle_self=False), pair_sum_odd(build([5]), False))

18 3 None


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi tới sinh đôi | lặp lại từ head | O(n²) | O(1) | minh họa, n nhỏ |
| 2. Chép ra mảng | truy cập hai đầu | O(n) | O(n) | **an toàn nhất**, không yêu cầu bộ nhớ |
| 3. Ngăn xếp | đẩy nửa đầu, pop khi đi nửa sau | O(n) | O(n/2) | không được sửa danh sách |
| 4. Đảo nửa sau | nhanh/chậm + đảo + khôi phục | O(n) | O(1) | yêu cầu O(1) bộ nhớ |
| 5. Đảo nửa đầu một lượt | đảo khi tìm giữa | O(n) | O(1) | muốn một lượt, code gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `pairSum([1, 2, 3, 4, 5, 6])` và `pairSum([9, 1, 1, 1, 1, 1, 1, 0])`?

**Bài 2 (tính tay):** Với `[1, 2, 3, 4, 5, 6]`, sau vòng tìm giữa của cách 4, `slow` ở nút có giá trị bao nhiêu? Sau khi đảo nửa sau, `to_list(head)` (đi từ head) cho ra gì?

**Bài 3 (điền TODO):** Viết `bt3_todo(head)` trả về **tổng sinh đôi nhỏ nhất** dùng ngăn xếp (cách 3).

**Bài 4 (tự viết lại):** Viết `bt4_todo(head)` cài cách 4 (đảo nửa sau) **có khôi phục** danh sách, trả về max twin sum.

**Bài 5 (mini challenge):** Viết `bt5_todo(head)` trả về **chỉ số `i`** (0 ≤ i < n/2) có tổng sinh đôi lớn nhất (hòa lấy nhỏ nhất), dùng O(1) bộ nhớ phụ.

In [32]:
def bt3_todo(head):
    # TODO: tổng sinh đôi nhỏ nhất bằng ngăn xếp
    pass

def bt4_todo(head):
    # TODO: đảo nửa sau, tính max, khôi phục danh sách
    pass

def bt5_todo(head):
    # TODO: chỉ số i có twin sum lớn nhất, O(1) bộ nhớ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: gập đôi. Mức 2: cặp `(1,6),(2,5),(3,4)` đều 7; cặp `(9,0)=9`, còn lại 2. Mức 3: `7` và `9`.

**Bài 2.** Mức 1: slow dừng ở nút `n/2 = 3` (chỉ số). Mức 2: giá trị 4. Mức 3: nửa đầu vẫn trỏ vào nút 4, mà nút 4 giờ có `next = None` ⇒ `[1, 2, 3, 4]`.

**Bài 3.** Mức 1: giống `stack_half`. Mức 2: thay `max` bằng `min`. Mức 3: khởi tạo `best = float('inf')`.

**Bài 4.** Mức 1: ba bước + khôi phục. Mức 2: lưu đầu nửa sau đã đảo `q`. Mức 3: cuối hàm `reverse(q)`.

**Bài 5.** Mức 1: như cách 4 nhưng đếm thêm chỉ số `k`. Mức 2: chỉ cập nhật khi `tổng > best` (lớn hơn hẳn). Mức 3: trả `best_k`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `7`, `9`. **Bài 2:** `slow.val = 4`; `to_list(head) = [1, 2, 3, 4]`.

In [33]:
assert Solution().pairSum(build([1, 2, 3, 4, 5, 6])) == 7
assert Solution().pairSum(build([9, 1, 1, 1, 1, 1, 1, 0])) == 9
h = build([1, 2, 3, 4, 5, 6]); slow = fast = h
while fast and fast.next: slow, fast = slow.next, fast.next.next
assert slow.val == 4
reverse(slow); assert to_list(h) == [1, 2, 3, 4]

def bt3(head):
    stack = []; slow = fast = head
    while fast and fast.next:
        stack.append(slow.val); slow, fast = slow.next, fast.next.next
    best = float("inf")
    while slow:
        best = min(best, slow.val + stack.pop()); slow = slow.next
    return best

def bt4(head):
    slow = fast = head
    while fast and fast.next: slow, fast = slow.next, fast.next.next
    q = reverse(slow); p, t, best = head, q, 0
    while t:
        best = max(best, p.val + t.val); p, t = p.next, t.next
    reverse(q)
    return best

def bt5(head):
    slow = fast = head
    while fast and fast.next: slow, fast = slow.next, fast.next.next
    q = reverse(slow); p, t = head, q
    best, best_k, k = -1, -1, 0
    while t:
        s = p.val + t.val
        if s > best: best, best_k = s, k
        p, t, k = p.next, t.next, k + 1
    reverse(q)
    return best_k

rng = random.Random(1930)
for _ in range(1500):
    n = 2 * rng.randint(1, 15); vals = [rng.randint(1, 20) for _ in range(n)]
    sums = [vals[i] + vals[~i] for i in range(n // 2)]
    assert bt3(build(vals)) == min(sums)
    hh = build(vals); assert bt4(hh) == max(sums) and to_list(hh) == vals
    hh = build(vals); assert bt5(hh) == sums.index(max(sums)) and to_list(hh) == vals
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
slow = fast = head
while fast and fast.next:            # slow -> đầu nửa sau (n chẵn)
    slow, fast = slow.next, fast.next.next
prev = None
while slow:                          # đảo nửa sau
    nxt = slow.next; slow.next = prev; prev = slow; slow = nxt
best = 0
while prev:                          # dừng theo nửa sau!
    best = max(best, head.val + prev.val)
    head, prev = head.next, prev.next
return best
```

**Checklist:**
- [ ] Vòng ghép dừng theo **nửa sau đã đảo** (`while q`).
- [ ] Đảo bằng vòng lặp, không đệ quy.
- [ ] Gán đồng thời: `slow.next` phải được gán trước `slow`.
- [ ] Sinh đôi là `n − 1 − i` (không phải `n − i`).
- [ ] Nếu cần giữ nguyên danh sách: khôi phục bằng một lần `reverse` nữa.